In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

train_path = "/content/drive/MyDrive/heart_disease_prediction/train.csv"
test_path = "/content/drive/MyDrive/heart_disease_prediction/test.csv"
sample_path = "/content/drive/MyDrive/heart_disease_prediction/sample_submission.csv"

df = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

print(df.shape)
df.head()

(223084, 20)


,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes


In [ ]:
target = "History of HeartDisease or Attack"
drop_cols = ["ID"]

X = df.drop(columns=drop_cols + [target])
y = df[target]

X_test = test.drop(columns=drop_cols)
y = y.map({'Yes':1, 'No':0})

In [ ]:
print(df.columns.tolist())

['ID', 'History of HeartDisease or Attack', 'High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked', 'Body Mass Index', 'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Leisure Physical Activity', 'Heavy Alcohol Consumption', 'Health Care Coverage', 'Doctor Visit Cost Barrier', 'General Health', 'Difficulty Walking', 'Sex', 'Education Level', 'Income Level', 'Age', 'Vegetable or Fruit Intake (1+ per Day)']


In [ ]:
print(y.isna().sum())

1694


In [ ]:
# map ก่อน
df[target] = df[target].map({'Yes':1, 'No':0})

# ลบแถวที่ target เป็น NaN
df = df[df[target].notna()]

# แยกใหม่
X = df.drop(columns=["ID", target])
y = df[target]

In [ ]:
print(y.isna().sum())

0


In [ ]:
y.value_counts(normalize=True)

,proportion
History of HeartDisease or Attack,
0.0,0.918388
1.0,0.081612


In [ ]:
neg = 0.918388
pos = 0.081612

scale_pos_weight = neg / pos
print(scale_pos_weight)

11.253100034308678


In [ ]:
binary_cols = [col for col in X.columns if set(X[col].dropna().unique()) <= {'Yes','No'}]
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = [col for col in X.columns if col not in binary_cols + num_cols]

print("binary:", binary_cols)
print("num:", num_cols)
print("cat:", cat_cols)

binary: ['High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked', 'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Leisure Physical Activity', 'Heavy Alcohol Consumption', 'Health Care Coverage', 'Doctor Visit Cost Barrier', 'Difficulty Walking', 'Vegetable or Fruit Intake (1+ per Day)']
num: ['Body Mass Index', 'Age']
cat: ['General Health', 'Sex', 'Education Level', 'Income Level']


In [ ]:
# ---------- BINARY ----------
for col in binary_cols:
    X[col] = X[col].map({'Yes':1, 'No':0})
    X_test[col] = X_test[col].map({'Yes':1, 'No':0})

    X[col] = X[col].fillna(-1)
    X_test[col] = X_test[col].fillna(-1)

# ---------- NUM ----------
for col in num_cols:
    X[col+'_missing'] = X[col].isna().astype(int)
    X_test[col+'_missing'] = X_test[col].isna().astype(int)

    median = X[col].median()
    X[col].fillna(median, inplace=True)
    X_test[col].fillna(median, inplace=True)

# ---------- CAT ----------
for col in cat_cols:
    X[col] = X[col].fillna("Unknown")
    X_test[col] = X_test[col].fillna("Unknown")

    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()

    all_data = pd.concat([X[col], X_test[col]])
    le.fit(all_data)

    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

/tmp/ipykernel_3212/1200830069.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X[col].fillna(median, inplace=True)
/tmp/ipykernel_3212/1200830069.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.m

In [ ]:
X['BMI_age'] = X['Body Mass Index'] * X['Age']
X_test['BMI_age'] = X_test['Body Mass Index'] * X_test['Age']

risk_cols = ['High Blood Pressure','Cholesterol','Smoked 100+ Cigarettes']
risk_cols = [c for c in risk_cols if c in X.columns]

X['risk_score'] = X[risk_cols].sum(axis=1)
X_test['risk_score'] = X_test[risk_cols].sum(axis=1)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    scale_pos_weight=11
)

model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 14454, number of negative: 162658
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004147 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 650
[LightGBM] [Info] Number of data points in the train set: 177112, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081609 -> initscore=-2.420679
[LightGBM] [Info] Start training from score -2.420679


LGBMClassifier(learning_rate=0.03, n_estimators=1000, scale_pos_weight=11)

In [ ]:
from sklearn.metrics import fbeta_score
import numpy as np

y_proba = model.predict_proba(X_val)[:,1]

best_score = 0
best_t = 0.5

for t in np.arange(0.2, 0.5, 0.01):
    y_pred = (y_proba > t).astype(int)
    score = fbeta_score(y_val, y_pred, beta=2)

    if score > best_score:
        best_score = score
        best_t = t

print("BEST:", best_t, best_score)

BEST: 0.49000000000000027 0.533487382629108


In [ ]:
y_test_proba = model.predict_proba(X_test)[:,1]
y_test_pred = pd.Series(y_test_pred).map({1:'Yes', 0:'No'})

sample[target] = y_test_pred
sample.to_csv("submission.csv", index=False)

In [ ]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>